# boolean-mask-combine — ex2: outlier-mask algebra via OR, AND-NOT, and XOR

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-combine`. Running the final beacon cell reports progress against the `Numpy: Boolean mask combine` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Boolean mask combine` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`boolean-mask-combine`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-combine"
DD_SUBTOPIC = "Numpy: Boolean mask combine"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Combining boolean masks — quick refresher

Three logical operators on bool tensors: `&`, `|`, `~`. All elementwise, all broadcast normally. PyTorch also exposes **XOR** via `^` — true iff exactly one operand is true.

```python
a ^ b   # symmetric difference
~(a | b)  # neither
a & ~b    # a but not b
```

The previous drill (ex1) ANDed five inside-test predicates into a single mask. This drill exercises the **OR / NOT / XOR** half of the algebra: combining several outlier-detection masks via OR (any outlier), AND-NOT (outlier on metric A but normal on B), and XOR (disagreement between two detectors).

### Exercise 2 — outlier-mask algebra via OR, AND-NOT, and XOR

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply elementwise `|`, `~`, and `^` to combine three per-row outlier predicates into (any-outlier, A-only-outlier, A-XOR-B disagreement) masks.
> Keywords: or, not, xor, outlier, set-algebra
> ```

**KCs targeted:** `mask-bitwise-and-or`, `mask-parenthesize-comparisons`

Implement `ex2_outlier_masks(x, threshold)`.

Given a 1-D tensor `x` of length `N` and a scalar `threshold`, build three boolean masks and return them as a tuple `(any_out, a_only, disagree)`:

1. **Detector A — magnitude:** `mask_a = x.abs() > threshold`.
2. **Detector B — sign-flip:** `mask_b = x < 0`.
3. **Detector C — non-finite:** `mask_c = ~t.isfinite(x)`.

Combine:
- `any_out  = mask_a | mask_b | mask_c`  (flagged by at least one detector)
- `a_only   = mask_a & ~mask_b & ~mask_c`  (flagged by A but not B and not C)
- `disagree = mask_a ^ mask_b`  (A and B disagree — XOR)

Each output must be `dtype=bool` and shape `(N,)`.

**Critical:** parenthesize every comparison. `&`, `|`, `^` bind tighter than `<`, `>`, `==` — `x > 0 | x < 1` parses as `x > (0 | x) < 1`.

In [ ]:
def ex2_outlier_masks(x: Tensor, threshold: float):
    mask_a = x.abs() > threshold
    mask_b = x < 0
    mask_c = ~t.isfinite(x)
    any_out = mask_a | mask_b | mask_c
    a_only = mask_a & ~mask_b & ~mask_c
    disagree = mask_a ^ mask_b
    return any_out, a_only, disagree


<details><summary>Solution</summary>

```python
def ex2_outlier_masks(x: Tensor, threshold: float):
    mask_a = x.abs() > threshold
    mask_b = x < 0
    mask_c = ~t.isfinite(x)
    any_out = mask_a | mask_b | mask_c
    a_only = mask_a & ~mask_b & ~mask_c
    disagree = mask_a ^ mask_b
    return any_out, a_only, disagree
```

**XOR `^` is the disagreement detector.** When you have two independent outlier signals, `a ^ b` highlights the rows where they disagree — these are the cases worth manually reviewing.

**`isfinite` is the canonical NaN/Inf check.** `x != x` works for NaN only; `~t.isfinite(x)` catches both NaN and ±inf in one shot.

**Difference from ex1.** ex1 built `mask & mask & mask & ...` — an intersection of *constraints*. ex2 builds `mask | mask | mask` (union) and `mask ^ mask` (symmetric difference), exercising the disjunctive half of the boolean algebra.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()